# Analysis and Visualization of Complex Agro-Environmental Data
---
## Main types of estimators

Here, very simple examples of application of commonly used estimation methods are shown, using simulated data.

##### Import modules:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, beta



#### Generate a dataset

In [ ]:
# Generate synthetic data (array of 30 samples from a normal distribution with mean 5 and standard deviation 2)
np.random.seed(24)
true_mu = 5
sigma = 2
data = np.random.normal(true_mu, sigma, size=30)

print("Data:", data)
print("True mean:", true_mu)

### 1. Maximum Likelihood Estimation (MLE) for mean of normal distribution

Maximum Likelihood Estimation (MLE) is a statistical method for estimating the unknown parameters of a probability distribution by maximizing a likelihood function. It finds the parameter values that make the observed data most probable.

In the simplest case of estimating a mean, if data are assumed to come from a normal distribution, the MLE finds the mean that maximizes a likelihood function based on the difference between data values and the Normal probability density function for a given value of 𝜇.

In [ ]:
# Define likelihood function for normal distribution with known sigma
def likelihood(mu, data, sigma):
    return np.prod(
        (1 / (np.sqrt(2 * np.pi) * sigma)) *
        np.exp(- (data - mu)**2 / (2 * sigma**2)) # This is the probability density function of a normal distribution
    )

# Evaluate likelihood for different mu values betwween 3 and 7
mu_values = np.linspace(3, 7, 10000) # Generate 10,000 values between 3 and 7 (we know that the true mean is 5, so we focus on that range)
likelihoods = [likelihood(mu, data, sigma) for mu in mu_values] # Calculate the likelihood for each mu value in the range

# Find MLE (maximum likelihood) estimate of mu
mle_mu = mu_values[np.argmax(likelihoods)]

print(f"MLE estimate of mu: {mle_mu:.3f}")

# Plot likelihood curve
plt.plot(mu_values, likelihoods)
plt.axvline(mle_mu, linestyle='--')
plt.title("Likelihood as a function of mu")
plt.xlabel("mu")
plt.ylabel("Likelihood")
plt.show()

Now let's compute the sample mean as usual:

In [ ]:
print(f"Sample mean: {np.mean(data):.3f}")

The sample mean estimated through MLE that assumes a normal distribution of data is equivalent to estimating the mean in the traditional way.

### 2. Ordinary Least Squares (OLS) for mean of normal distribution

Ordinary Least Squares (OLS) is used to estimate unknown parameters (usually of a model) by minimizing the sum of the squared differences (residuals) between observed data values and the predicted value.

In [ ]:
# Try different mu values and calculate the sum of squared errors (OLS)
mu_values = np.linspace(3, 7, 10000) # Generate 10,000 evenly spaced numbers between 3 and 7 
errors = [np.sum((data - mu)**2) for mu in mu_values] # Calculate the sum of squared errors for each mu value

# Find best mu (OLS)
ols_mu = mu_values[np.argmin(errors)] # Find the mu value that minimizes the sum of squared errors

print(f"OLS estimate of mu: {ols_mu:.3f}") 

Now let's compute the sample mean as usual:

In [ ]:
print(f"Sample mean: {np.mean(data):.3f}")

In fact the sample mean is the OLS estimate of mu for a normal distribution with known variance! The estimated meanshould be close to the true mean of 5, but it may not be exactly the same due to the randomness in the data.

### 3. Bootstrap estimation

Bootstrap estimation is a powerful, nonparametric resampling technique used in statistics to estimate the sampling distribution of a statistic (such as the mean or median) by repeatedly sampling with replacement from a single original dataset. It calculates variability, confidence intervals, and standard errors without assuming normality.

Here, we will use bootstrap estimation to estimate the mean and standard deviation of the previously generated data (note that each time you run the bootstrap estimation, the results will slightly differ)

In [ ]:
# Bootstrap estimation of mean and standard error
n_boot = 1000 # Number of bootstrap samples
bootstrap_means = [] # List to store bootstrap means

for _ in range(n_boot):
    sample = np.random.choice(data, size=len(data), replace=True) # Generate a bootstrap sample by sampling with replacement from the original data
    bootstrap_means.append(np.mean(sample)) # Calculate the mean of the bootstrap sample and store it

bootstrap_means = np.array(bootstrap_means) # Convert list to numpy array for easier calculations

print(f"Bootstrap estimate of mean: {np.mean(bootstrap_means):.3f}") # Calculate the mean of the bootstrap means to get the bootstrap estimate of the mean
print(f"Bootstrap std (SE): {np.std(bootstrap_means):.3f}") # Calculate the standard deviation of the bootstrap means to get the bootstrap estimate of the standard error

# Plot bootstrap distribution of the mean
plt.figure()
plt.hist(bootstrap_means, bins=30)
plt.axvline(ols_mu, linestyle='--')
plt.title("Bootstrap distribution of the mean")
plt.show()

The estimated mean is close to the mean value estimated in the traditional way. Bootstrap allows also to get an estimate of the sample mean standard deviation. 

IMPORTANT NOTE: Bootstrap estimates the standard deviation of the sample mean not the standard deviation of the population (which should be close to 3)!

### 4. Jackknife estimation


Jackknife estimation is a resampling technique used to measure the bias and standard error of a statistic by systematically leaving out one observation at a time from the dataset. By recalculating the estimator 
 times (for samples of size ***n-1***), it provides a non-parametric way to estimate variance and improve accuracy without needing strict distribution assumptions.

Using the same generated dataset:

In [ ]:
# Jackknife estimation of mean and standard error
n = len(data) # Number of samples (equal to the number of jackknife samples)
jackknife_means = [] # List to store jackknife means (each mean is calculated by leaving out one sample from the original data)

for i in range(n):
    sample = np.delete(data, i) # Create a jackknife sample by deleting the i-th sample from the original data
    jackknife_means.append(np.mean(sample)) # Calculate the mean of the jackknife sample and store it

jackknife_means = np.array(jackknife_means) # Convert list to numpy array for easier calculations

# Jackknife estimate (usually same as mean)
jackknife_mean = np.mean(jackknife_means) # Calculate the mean of the jackknife means to get the jackknife estimate of the mean

# Jackknife standard error
jackknife_se = np.sqrt((n - 1) * np.mean((jackknife_means - jackknife_mean)**2)) # Calculate the jackknife standard error using the formula: SE = sqrt((n-1) * mean((theta_i - theta_jackknife)^2))

print(f"Jackknife estimate: {jackknife_mean:.3f}")
print(f"Jackknife SE: {jackknife_se:.3f}")

# Plot jackknife distribution
plt.figure()
plt.hist(jackknife_means, bins=30)
plt.axvline(ols_mu, linestyle='--')
plt.title("Jacknife distribution of the mean")
plt.show()

The estimated mean is close to the mean value estimated in the traditional way. The SE is the standard error of the estimated mean, but it is also a measure of overall sensitive of an estimator to each observation.

### 5. Bayesian Estimation


Bayesian estimation is a statistical method that calculates the probability of a parameter (like a mean or probability) by combining prior knowledge or beliefs (prior distribution) with new evidence (likelihood) using Bayes' theorem. It treats parameters as random variables, yielding a posterior distribution that is updated as more evidence becomes available.

Let's use the same example as before: estimate of the mean of the generated sample

In [ ]:
# Bayesian inference (normal mean, known sigma)
# Prior: mu ~ Normal(mu0, tau^2)
mu0 = 0 # Prior mean (we can choose any value, but we will use 0 for simplicity)
tau = 5 # Prior standard deviation

# Posterior parameters
posterior_variance = 1 / (n / sigma**2 + 1 / tau**2) # The posterior variance is calculated using the formula for the normal distribution with known variance: 1 / (n/sigma^2 + 1/tau^2)
posterior_mean = posterior_variance * (np.sum(data) / sigma**2 + mu0 / tau**2) # The posterior mean is calculated using the formula: posterior_variance * (sum(data)/sigma^2 + mu0/tau^2)

print("\nBayesian posterior mean:", posterior_mean)

# Plot posterior
x = np.linspace(0, 10, 200) # Generate 200 evenly spaced numbers between 0 and 10 to plot the posterior distribution
posterior_pdf = norm.pdf(x, posterior_mean, np.sqrt(posterior_variance)) # Calculate the posterior probability density function using the normal distribution with mean equal to the posterior mean and standard deviation equal to the square root of the posterior variance

plt.figure()
plt.plot(x, posterior_pdf)
plt.axvline(ols_mu, linestyle='--')
plt.title("Posterior distribution of mu")
plt.show()



Here, the posterior mean (estimate) is a compromise between what the data say and what we believed before. 
Now let's use a more interactive code, to understand better the effect of selecting different sample sizes and prior standard deviation.

In [ ]:
# --- Try changing these ---
n = 30              # sample size (try 5, 30, 100)
sigma = 2          # known data std
mu0 = 0            # prior mean
tau = 1            # prior std (try small=strong prior, large=weak prior)
# -------------------------

np.random.seed(24)
true_mu = 5
data2 = np.random.normal(true_mu, sigma, n)

# Sample mean
x_bar = np.mean(data2)

# Posterior parameters
posterior_variance = 1 / (n / sigma**2 + 1 / tau**2)
posterior_mean = posterior_variance * (np.sum(data2) / sigma**2 + mu0 / tau**2)

# Plot
x = np.linspace(-2, 10, 200)

prior = norm.pdf(x, mu0, tau)
posterior = norm.pdf(x, posterior_mean, np.sqrt(posterior_variance))

plt.figure()
plt.plot(x, prior, linestyle='--', label='Prior')
plt.plot(x, posterior, label='Posterior')
plt.axvline(x_bar, linestyle=':', label='Sample mean (MLE)')
plt.title("Bayesian Updating of μ")
plt.legend()
plt.show()

print(f"Sample mean (MLE): {x_bar:.4f}")
print(f"Posterior mean: {posterior_mean:.4f}")

In conclusion:

1. When you increase ***n*** posterior moves toward the sample mean

2. When you increase ***tau*** the prior gets weaker and posterior moves toward the sample mean

Now let's to a summary of all the estimates we have calculated:

In [ ]:
# Summary
print("\nSummary:")
print("True mean:", true_mu)
print(f"MLE estimate of mu: {mle_mu:.3f}")
print(f"OLS estimate of mu: {ols_mu:.3f}") 
print(f"Bootstrap estimate of mean: {np.mean(bootstrap_means):.3f}") #
print(f"Jackknife estimate of mean: {jackknife_mean:.3f}")
print(f"Bayesian mean: {posterior_mean:.3f}")
